# 🎙️ Speech Intelligence Engine — Google Colab GPU Trainer

This notebook offloads heavy **Speech Foundation Model** training, SSL feature extraction (WavLM / HuBERT), and Speech-LLM adapter fine-tuning to Google Colab GPUs (T4 / L4 / A100), persisting all checkpoints directly to your 5TB **Google Drive** (`rakshithgoud453@gmail.com`).

## Step 1: Check GPU & Mount Google Drive (5TB Storage)

In [1]:
# Check available GPU resources
!nvidia-smi

# Mount 5TB Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')

for path in [DRIVE_BASE_PATH, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"✓ Ready directory on Google Drive: {path}")

Fri Oct  2 06:21:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install ML & Audio Dependencies (Colab-Optimized)

In [2]:
# Install extra NLP, Audio & ML packages with numpy<2.0.0 pin for Colab compatibility
!pip install -q "numpy<2.0.0" transformers datasets peft accelerate bitsandbytes librosa soundfile wandb pyannote.audio faster-whisper hdbscan


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.

## Step 3: Clone / Sync Repository Code

In [3]:
# Codebase setup in Colab workspace
import os

REPO_URL = "https://github.com/rakshithgoud453/speech_intelligence_engine.git"
WORKSPACE_DIR = "/content/speech_intelligence_engine"

%cd /content
if os.path.exists(WORKSPACE_DIR):
    %cd {WORKSPACE_DIR}
    !git pull
else:
    !git clone {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

print(f"✓ Currently operating in: {os.getcwd()}")

/content
Cloning into '/content/speech_intelligence_engine'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 127 (delta 49), reused 105 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 477.00 KiB | 1.80 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/speech_intelligence_engine
✓ Currently operating in: /content/speech_intelligence_engine


## Step 4: SSL Layer Representation Extraction (WavLM / HuBERT)

In [4]:
import torch
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Pre-trained SSL Model natively via Torchaudio
bundle = torchaudio.pipelines.WAVLM_BASE
ssl_model = bundle.get_model().to(device)
ssl_model.eval()

def extract_ssl_representations(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != bundle.sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=bundle.sample_rate)
        waveform = resampler(waveform)

    with torch.no_grad():
        all_layers, _ = ssl_model.extract_features(waveform.to(device))

    # Layers 4-8: Prosodic/Phonetic representations
    prosodic_rep = torch.stack(all_layers[4:9]).mean(dim=0).squeeze(0)
    # Layers 9-11: Semantic representations
    semantic_rep = torch.stack(all_layers[9:12]).mean(dim=0).squeeze(0)

    return {
        "prosodic_embedding": prosodic_rep.cpu().numpy(),
        "semantic_embedding": semantic_rep.cpu().numpy()
    }

print("✓ SSL Feature Extraction Pipeline Loaded Successfully.")


## Step 5: Checkpoint Saver Utility

In [ ]:
import os
import time
import torch

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)

    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)

    print(f"✓ Saved Checkpoint to Google Drive (5TB): {save_path}")

print(f"✓ Checkpoints directory configured: {CHECKPOINT_DIR}")

## Step 6: Batch Audio Ingestion & Model Checkpoint Execution Loop 🚀 (Self-Contained & Auto-Logging)

In [ ]:
import sys
import glob
import json
import os
import time
import traceback
import soundfile as sf
import numpy as np
import torch

# 1. Ensure Google Drive Output Directories
DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')
for p in [CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(p, exist_ok=True)

# 2. Local Saver Fallback
def local_save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)
    print(f"✓ Saved Checkpoint to 5TB Google Drive: {save_path}")

# 3. Execute Loop with Automated Error Logging
try:
    WORKSPACE_DIR = "/content/speech_intelligence_engine"
    for p in [WORKSPACE_DIR, os.path.dirname(WORKSPACE_DIR)]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        from speech_intelligence_engine import SpeechLearningEngine
    except ImportError:
        from engine import SpeechLearningEngine

    try:
        from speech_intelligence_engine.representation.projector import SpeechLLMProjector
    except ImportError:
        from representation.projector import SpeechLLMProjector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Initializing Engine & Projector on device: {device}...")

    engine = SpeechLearningEngine(model_type="wavlm", replay_capacity=1000)
    projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=4096).to(device)

    audio_files = glob.glob(os.path.join(DATASET_DIR, "*.wav")) + glob.glob(os.path.join(DATASET_DIR, "*.mp3"))

    if not audio_files:
        print(f"No audio files in Google Drive datasets. Generating benchmark audio sample...")
        sample_wav = os.path.join(DATASET_DIR, "sample_speech_benchmark.wav")
        t = np.linspace(0, 3.0, int(16000 * 3.0), endpoint=False)
        sig = (0.5 * np.sin(2 * np.pi * 180 * t) + 0.2 * np.sin(2 * np.pi * 360 * t)).astype(np.float32)
        sf.write(sample_wav, sig, 16000)
        audio_files = [sample_wav]

    for i, audio_path in enumerate(audio_files, 1):
        print(f"\n[Processing {i}/{len(audio_files)}]: {audio_path}")
        res = engine.assimilate(audio_path, recording_id=os.path.basename(audio_path))

        metrics = {
            "episodes": res.num_episodes,
            "voiced_segments": res.num_segments,
            "novelty_rate": res.novelty_rate,
            "duration_sec": res.duration_sec
        }
        local_save_checkpoint(projector, step=i, metrics=metrics)

    status_payload = {
        "status": "COMPLETED",
        "total_episodes_learned": engine.get_state().total_episodes,
        "prototypes_discovered": engine.get_state().total_prototypes,
        "checkpoint_directory": CHECKPOINT_DIR,
        "last_updated": time.ctime()
    }

    status_file = os.path.join(OUTPUT_DIR, "status.json")
    with open(status_file, "w") as f:
        json.dump(status_payload, f, indent=2)

    print(f"\n✨ Batch processing complete!")
    print(f"📁 Checkpoint saved to Google Drive: {CHECKPOINT_DIR}")
    print(f"📄 Status summary written to: {status_file}")

except Exception as e:
    err_trace = traceback.format_exc()
    print(f"\n❌ EXECUTION ERROR IN STEP 6:\n{err_trace}")
    error_log_path = os.path.join(OUTPUT_DIR, "error_log.txt")
    with open(error_log_path, "w") as f:
        f.write(err_trace)
    print(f"📄 Full Error Traceback written to Google Drive: {error_log_path}")
    raise e

## Step 7: Native Speech-LLM LoRA Adapter Alignment & Multimodal Inference 🧠

In this step, raw WavLM 768-d continuous acoustic/prosodic latents are projected directly into LLM token space (), allowing a lightweight LoRA-adapted LLM backbone to reason over speech *without* cascaded text transcription.

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from speech_intelligence_engine.representation.projector import SpeechLLMProjector

DRIVE_BASE_PATH = "/content/drive/MyDrive/speech_intelligence_engine"
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, "checkpoints")
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, "outputs")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LLM_MODEL_ID = "Qwen/Qwen1.5-0.5B"
print(f"🚀 Initializing Speech-LLM Multimodal Adapter ({LLM_MODEL_ID})...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
llm = get_peft_model(llm, lora_config)
llm.print_trainable_parameters()

projector_dim = llm.config.hidden_size
projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=projector_dim).to(device)

speech_latents = torch.randn(1, 50, 768, device=device)
projected_tokens = projector(speech_latents)

llm_embeddings = llm.get_input_embeddings()
text_input = "Analyze spoken input:"
text_tokens = tokenizer(text_input, return_tensors="pt").input_ids.to(device)
text_embeds = llm_embeddings(text_tokens)

multimodal_embeds = torch.cat([projected_tokens.to(text_embeds.dtype), text_embeds], dim=1)

with torch.no_grad():
    output_ids = llm.generate(inputs_embeds=multimodal_embeds, max_new_tokens=30)
    response_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("
✨ Speech-LLM Multimodal Forward Pass Verified!")
print(f"🎙️ Speech Latent Tokens: {projected_tokens.shape}")
print(f"💬 Native LLM Multimodal Response: '{response_text.strip()}'")

adapter_save_path = os.path.join(CHECKPOINT_DIR, "speech_llm_lora_adapter")
llm.save_pretrained(adapter_save_path)
torch.save(projector.state_dict(), os.path.join(adapter_save_path, "speech_projector.pt"))
print(f"✓ Saved Speech-LLM LoRA Adapter & Projector to Google Drive: {adapter_save_path}")
